In [40]:
import joblib
import os
import time
import matplotlib.pyplot as plt
import csv
from typing import Literal
import pandas as pd
import numpy as np
import seaborn as sns
from scipy import stats
from scipy.stats import iqr
from datetime import date

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import root_mean_squared_log_error
from sklearn.model_selection import RandomizedSearchCV

from xgboost import XGBRegressor
from catboost import CatBoostRegressor

df_treino = pd.read_csv("./data/treino_norm.csv",keep_default_na=False)

arquivo = "comparacao_baseline_modelos_treino.csv"

if os.path.exists(arquivo):
    os.remove(arquivo)

def transformar_dolar(valor:int):
    return np.expm1(valor)

def salvar_modelo(modelo,caminho:str):
    
    os.makedirs("./modelos",exist_ok=True)
    
    joblib.dump(modelo,caminho)

def criar_csv_comparacao(modelo, rmsle, rmse, mae, r2, tempo_s):
    arquivo_existe = os.path.exists(arquivo)

    with open(arquivo, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        if not arquivo_existe:
            writer.writerow(["modelo", "rmsle", "rmse", "mae", "r2","tempo_s"])

        writer.writerow([modelo, f"{rmsle:.4f}", f"{rmse:.4f}", f"{mae:.4f}", f"{r2:.4f}", f"{tempo_s:.4f}"])

In [41]:
def comparar_baseline(alvo: Literal["RMSLE", "RMSE", "MAE", "R2"], metrica):
    RMSLE_META = 0.17543
    RMSE_META = 36061.40
    MAE_META = 22186.99
    R2_META = 0.83046

    match alvo:
        case "RMSLE":
            return metrica < RMSLE_META
        case "RMSE":
            return metrica < RMSE_META 
        case "MAE":
            return metrica < MAE_META  
        case "R2":
            return metrica > R2_META

# RandomForestRegressor

In [42]:
def treinar_random_forest(df:pd.DataFrame):
    X = df.drop(columns=["SalePrice","Id"])
    Y = df["SalePrice"]

    colunas_numericas = X.select_dtypes(include=["int64","float64"]).columns
    # colunas_categoricas = X.select_dtypes(include=["str"]).columns
    colunas_categoricas = X.select_dtypes(include=["object","category","string"]).columns

    transformador_numerico = Pipeline(
        steps= [
            ("imputer",SimpleImputer(strategy="median"))
        ]        
    )

    transformador_categorico = Pipeline(
        steps= [
            ("imputer",SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]        
    )

    pre_processador = ColumnTransformer(
        transformers=[
            ("numerico",transformador_numerico, colunas_numericas),
            ("categorico",transformador_categorico,colunas_categoricas)
        ]
    )

    X_treino, X_teste, Y_treino, Y_teste = train_test_split(
        X,
        Y,
        test_size=0.2,
        random_state=42,
    )

    pipeline = Pipeline(
        steps=[
            ("pre-processamento",pre_processador),
            ("regressao",RandomForestRegressor(
                random_state=42,
                n_jobs=-1
            ))
        ]
    )

    hiperparametros = {
        "regressao__n_estimators": [50, 100],
        "regressao__max_depth": [None,20],
        "regressao__min_samples_split": [2, 10],
        "regressao__min_samples_leaf": [1,4],
        "regressao__max_features": ["sqrt", "log2"]
    }

    modelo = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=hiperparametros,
        n_iter=15,
        cv=3,
        scoring="neg_root_mean_squared_error",
        verbose=1,
        random_state=42,
        n_jobs=-1
    )

    tempo_comeco = time.perf_counter()
    modelo.fit(X_treino,Y_treino)
    tempo_fim = time.perf_counter()

    tempo_total = tempo_fim - tempo_comeco

    Y_predito = modelo.predict(X_teste)

    Y_teste_real = transformar_dolar(Y_teste)
    Y_predito_real = transformar_dolar(Y_predito)

    rmsle = np.sqrt(mean_squared_error(Y_teste, Y_predito))
    rmse = mean_squared_error(Y_teste_real,Y_predito_real) ** 0.5
    mae = mean_absolute_error(Y_teste_real,Y_predito_real)
    r2 = r2_score(Y_teste_real,Y_predito_real)

    salvar_modelo(modelo,"./modelos/random-forest-baseline.joblib")

    criar_csv_comparacao("random_florest",rmsle,rmse,mae,r2,tempo_total)

    print(f"melhores hiperparametros: {modelo.best_params_}")
    print(f"melhor score: {modelo.best_score_:.4f}")
    # print(f"{modelo.best_estimator_}")

    print(f"RMSLE: $ {rmsle:.4f} - {comparar_baseline("RMSLE",rmsle)}")
    print(f"RMSE: {rmse:.4f} - {comparar_baseline("RMSE",rmse)}")
    print(f"MAE: $ {mae:.4f} - {comparar_baseline("MAE",mae)}")
    print(f"R2: $ {r2:.4f} - {comparar_baseline("R2",r2)}")

print("random florest")
treinar_random_forest(df_treino)

random florest
Fitting 3 folds for each of 15 candidates, totalling 45 fits


melhores hiperparametros: {'regressao__n_estimators': 100, 'regressao__min_samples_split': 2, 'regressao__min_samples_leaf': 1, 'regressao__max_features': 'sqrt', 'regressao__max_depth': 20}
melhor score: -0.1307
RMSLE: $ 0.1292 - True
RMSE: 27148.5363 - True
MAE: $ 16992.0104 - True
R2: $ 0.8190 - False


# XGBoost

In [43]:

def treinar_xgboost(df:pd.DataFrame):
    X = df.drop(columns=["SalePrice","Id"])
    Y = df["SalePrice"]

    colunas_numericas = X.select_dtypes(include=["int64","float64"]).columns
    colunas_categoricas = X.select_dtypes(include=["object","category","string"]).columns

    transformador_numerico = Pipeline(
        steps= [
            ("imputer",SimpleImputer(strategy="median"))
        ]        
    )

    transformador_categorico = Pipeline(
        steps= [
            ("imputer",SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]        
    )

    pre_processador = ColumnTransformer(
        transformers=[
            ("numerico",transformador_numerico, colunas_numericas),
            ("categorico",transformador_categorico,colunas_categoricas)
        ]
    )

    X_treino, X_teste, Y_treino, Y_teste = train_test_split(
        X,
        Y,
        test_size=0.2,
        random_state=42,
    )


    pipeline = Pipeline(
        steps=[
            ("pre-processamento",pre_processador),
            ("regressao",XGBRegressor(
                random_state=42
            ))
        ]
    )
    
    hiperparametros = {
        "regressao__n_estimators": [200, 500, 800],
        "regressao__learning_rate": [0.01,0.05, 0.1],
        "regressao__max_depth": [3,6, 8],
        "regressao__min_child_weight": [2,5],
        "regressao__subsample": [0.2, 0.5, 1.0],
        "regressao__colsample_bytree": [0.5,1.0],
        "regressao__gamma": [0.1,0.5,0.1],
        "regressao__reg_lambda": [0.5,1]
    }

    modelo = RandomizedSearchCV(
        estimator=pipeline,
        param_distributions=hiperparametros,
        n_iter=15,
        cv=5,
        scoring="neg_root_mean_squared_error",
        verbose=1,
        random_state=42,
        n_jobs=-1
    )
    
    tempo_comeco = time.perf_counter()
    modelo.fit(X_treino,Y_treino)
    tempo_fim = time.perf_counter()

    tempo_total = tempo_fim - tempo_comeco

    Y_predito = modelo.predict(X_teste)

    Y_teste_real = transformar_dolar(Y_teste)
    Y_predito_real = transformar_dolar(Y_predito)

    rmsle = np.sqrt(mean_squared_error(Y_teste, Y_predito))
    rmse = mean_squared_error(Y_teste_real,Y_predito_real) ** 0.5
    mae = mean_absolute_error(Y_teste_real,Y_predito_real)
    r2 = r2_score(Y_teste_real,Y_predito_real)

    salvar_modelo(modelo,"./modelos/xgboost-baseline.joblib")
    criar_csv_comparacao("xgboost",rmsle,rmse,mae,r2,tempo_total)

    print(f"melhores hiperparametros: {modelo.best_params_}")
    print(f"melhor score: {modelo.best_score_:.4f}")
    print(f"{modelo.best_estimator_}")

    print(f"RMSLE: $ {rmsle:.4f} - {comparar_baseline("RMSLE",rmsle)}")
    print(f"RMSE: {rmse:.4f} - {comparar_baseline("RMSE",rmse)}")
    print(f"MAE: $ {mae:.4f} - {comparar_baseline("MAE",mae)}")
    print(f"R2: $ {r2:.4f} - {comparar_baseline("R2",r2)}")

print("xgboost")
treinar_xgboost(df_treino)

xgboost
Fitting 5 folds for each of 15 candidates, totalling 75 fits
melhores hiperparametros: {'regressao__subsample': 0.5, 'regressao__reg_lambda': 0.5, 'regressao__n_estimators': 800, 'regressao__min_child_weight': 2, 'regressao__max_depth': 3, 'regressao__learning_rate': 0.05, 'regressao__gamma': 0.1, 'regressao__colsample_bytree': 0.5}
melhor score: -0.1119
Pipeline(steps=[('pre-processamento',
                 ColumnTransformer(transformers=[('numerico',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median'))]),
                                                  Index(['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond',
       'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2',
       'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'LowQualFinSF',
       'GrLi...
                              feature_types=None, feature

# CatBoost

In [44]:
def treinar_catboost(df:pd.DataFrame):
    X = df.drop(columns=["SalePrice","Id"])
    Y = df["SalePrice"]

    colunas_numericas = X.select_dtypes(include=["int64","float64"]).columns
    colunas_categoricas = X.select_dtypes(include=["object","category","string","str"]).columns.tolist()

    # transformador_numerico = Pipeline(
    #     steps= [
    #         ("imputer",SimpleImputer(strategy="median"))
    #     ]        
    # )

    # transformador_categorico = Pipeline(
    #     steps= [
    #         ("imputer",SimpleImputer(strategy="most_frequent"))
    #         ("onehot", OneHotEncoder(handle_unknown="ignore"))
    #     ]        
    # )

    # pre_processador = ColumnTransformer(
    #     transformers=[
    #         ("numerico",transformador_numerico, colunas_numericas),
    #         ("categorico",transformador_categorico,colunas_categoricas)
    #     ]
    # )

    X_treino, X_teste, Y_treino, Y_teste = train_test_split(
        X,
        Y,
        test_size=0.2,
        random_state=42,
    )


    catboost = CatBoostRegressor(
        verbose=0,
        thread_count=2,
        random_state=42,
        early_stopping_rounds=50
    )
    
    hiperparametros = {
        "iterations": [50, 100],
        "learning_rate": [0.01,0.05,0.1],
        "depth": [2,5],
        "l2_leaf_reg": [1,5,8],
        "random_strength": [1,5,10],
        "min_data_in_leaf": [1,5],
        "bagging_temperature": [0,3],
        "loss_function": ["RMSE"]
    }

    modelo = RandomizedSearchCV(
        estimator=catboost,
        param_distributions=hiperparametros,
        n_iter=15,
        cv=5,
        scoring="neg_root_mean_squared_error",
        verbose=1,
        random_state=42,
        n_jobs=2
    )
    
    tempo_comeco = time.perf_counter()
    modelo.fit(X_treino,Y_treino,cat_features=colunas_categoricas)
    tempo_fim = time.perf_counter()

    tempo_total = tempo_fim - tempo_comeco

    Y_predito = modelo.predict(X_teste)

    Y_teste_real = transformar_dolar(Y_teste)
    Y_predito_real = transformar_dolar(Y_predito)

    rmsle = np.sqrt(mean_squared_error(Y_teste, Y_predito))
    rmse = mean_squared_error(Y_teste_real,Y_predito_real) ** 0.5
    mae = mean_absolute_error(Y_teste_real,Y_predito_real)
    r2 = r2_score(Y_teste_real,Y_predito_real)

    salvar_modelo(modelo,"./modelos/catboost-baseline.joblib")
    criar_csv_comparacao("catboost",rmsle,rmse,mae,r2,tempo_total)

    print(f"melhores hiperparametros: {modelo.best_params_}")
    print(f"melhor score: {modelo.best_score_:.4f}")
    # print(f"{modelo.best_estimator_}")

    print(f"RMSLE: $ {rmsle:.4f} - {comparar_baseline("RMSLE",rmsle)}")
    print(f"RMSE: {rmse:.4f} - {comparar_baseline("RMSE",rmse)}")
    print(f"MAE: $ {mae:.4f} - {comparar_baseline("MAE",mae)}")
    print(f"R2: $ {r2:.4f} - {comparar_baseline("R2",r2)}")

print("catboost")
treinar_catboost(df_treino)

catboost
Fitting 5 folds for each of 15 candidates, totalling 75 fits
melhores hiperparametros: {'random_strength': 1, 'min_data_in_leaf': 5, 'loss_function': 'RMSE', 'learning_rate': 0.1, 'l2_leaf_reg': 5, 'iterations': 100, 'depth': 5, 'bagging_temperature': 0}
melhor score: -0.1138
RMSLE: $ 0.1181 - True
RMSE: 24321.6726 - True
MAE: $ 15612.8628 - True
R2: $ 0.8547 - True


# Resultados
